In [ ]:
"""
OVERVIEW:
    This script implements a robust, object-oriented pipeline for binary sentiment classification.
    It integrates classical machine learning (SVM, Logistic Regression) with deep learning
    feature extraction techniques (Word2Vec, Doc2Vec, BERT) and optimizes performance using
    Optuna for hyperparameter tuning.

KEY ARCHITECTURAL DECISIONS:
    1. Feature Engineering Strategy:
       - BERT: Uses the [CLS] token embedding directly. This is the intended usage for
         classification tasks in the original BERT paper and typically yields stronger
         semantic signals for sentence-level tasks than mean pooling.
       - Doc2Vec: Trained for 50 epochs to ensure vector space convergence on smaller datasets.
       - Global Caching: Implements a global cache mechanism for BERT/GloVe to avoid
         redundant computations during iterative tuning.

    2. Preprocessing & Optimization:
       - StandardScaler: CRITICAL. When using the manual Hinge Loss SGD classifier with
         dense vectors (Word2Vec/Doc2Vec), features are standardized (mean=0, var=1).
         This ensures the Stochastic Gradient Descent converges efficiently and prevents
         gradient explosion/vanishing.

    3. Model Selection:
       - Implements a custom 'ManualHingeSGD' class to demonstrate algorithmic understanding.
       - Uses LinearSVC and standard SVC with RBF kernels based on parameter search.

USAGE:
    Run the script directly. It will:
    1. Download necessary models (BERT/GloVe) from mirror sites.
    2. Load training/testing data.
    3. Iterate through combinations of Features (Count, TF-IDF, W2V, BERT) and Classifiers.
    4. Perform Bayesian Hyperparameter Optimization (Optuna).
    5. Output performance metrics (Accuracy, F1, AUC) and save plots.
"""

import os
import sys

# ==============================================================================
# 1. Environment Setup
# ==============================================================================
# Core Configuration: Set mirror sources before importing transformers/torch
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["TRANSFORMERS_OFFLINE"] = "0"  # Ensure online mode
os.environ["HF_HUB_OFFLINE"] = "0"

# Fallback mirror options
# os.environ["HF_ENDPOINT"] = "https://mirrors.tuna.tsinghua.edu.cn/hugging-face-models"
# os.environ["HF_ENDPOINT"] = "https://mirror.sjtu.edu.cn/huggingface"

# ==============================================================================
# 2. Imports
# ==============================================================================
import random
import time
import warnings
import gc
from abc import ABC, abstractmethod

# Data Science Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from tqdm import tqdm
import optuna

# Machine Learning & NLP Libraries
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

# Deep Learning & NLP Specifics
import torch
import gensim.downloader as api
from gensim.models import Word2Vec
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

# Import transformers with error handling
try:
    from transformers import BertTokenizer, BertModel

    TRANSFORMERS_AVAILABLE = True
except Exception as e:
    print(f"[Warning] Transformers import failed: {e}")
    TRANSFORMERS_AVAILABLE = False

# ==============================================================================
# 3. Global Configuration
# ==============================================================================
# System Config
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 2025
MODEL_SAVE_DIR = "saved_models"
PLOT_SAVE_DIR = "plots"

# Suppress warnings
warnings.filterwarnings("ignore")

# Set Random Seeds for Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Create output directories
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
os.makedirs(PLOT_SAVE_DIR, exist_ok=True)

# Global Cache Variables
GLOBAL_GLOVE_VECTORS = None
GLOBAL_BERT_MODEL = None
GLOBAL_BERT_TOKENIZER = None
GLOBAL_BERT_TRAIN_EMBEDDINGS = None
GLOBAL_BERT_TEST_EMBEDDINGS = None

print("System Configuration:")
print(f"Compute Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"Transformers Available: {TRANSFORMERS_AVAILABLE}")
print("-" * 60)


# ==============================================================================
# 4. Utility Functions
# ==============================================================================
def download_all_pretrained_models():
    """
    Downloads all necessary pre-trained models at startup using mirrors.
    Gracefully handles failures and provides fallback options.
    """
    print("\n>>> 0. Pre-downloading Models from Mirrors...")

    # 1. Download BERT (with multiple fallback strategies)
    if TRANSFORMERS_AVAILABLE:
        print("  [BERT] Checking/Downloading 'bert-base-uncased'...")
        try:
            # Strategy 1: Try with current mirror setting
            tokenizer = BertTokenizer.from_pretrained(
                "bert-base-uncased", cache_dir="./cache", force_download=False
            )
            model = BertModel.from_pretrained(
                "bert-base-uncased", cache_dir="./cache", force_download=False
            )

            # Store in global cache
            global GLOBAL_BERT_TOKENIZER, GLOBAL_BERT_MODEL
            GLOBAL_BERT_TOKENIZER = tokenizer
            GLOBAL_BERT_MODEL = model.to(DEVICE)
            GLOBAL_BERT_MODEL.eval()

            print("  [BERT] Successfully loaded and cached.")

        except Exception as e:
            print(f"  [BERT] Download failed: {e}")
            print("  [Info] BERT will be skipped in experiments.")
            print("  [Hint] You can:")
            print("         1. Check your network connection")
            print("         2. Try different mirror in line 9-11")
            print("         3. Manually download to './cache' directory")
    else:
        print("  [BERT] Skipped (transformers not available)")

    # 2. Download GloVe
    print("  [GloVe] Checking/Downloading 'glove-wiki-gigaword-300'...")
    global GLOBAL_GLOVE_VECTORS
    try:
        GLOBAL_GLOVE_VECTORS = api.load("glove-wiki-gigaword-300")
        print("  [GloVe] Successfully loaded into memory.")
    except Exception as e:
        print(f"  [GloVe] Download failed: {e}")
        print("  [Info] Word2Vec will fallback to training from scratch.")


def read_examples(path):
    """Reads data from the specified path."""
    examples = []
    if not os.path.exists(path):
        print(f"[Error] File not found: {path}")
        return []

    with open(path, encoding="ISO-8859-1") as f:
        for line in f:
            parts = line.split(" ", 1)
            if len(parts) == 2:
                y, x = parts
                examples.append((x.strip(), int(y)))
    print(f"[Data Load] Read {len(examples)} examples from {path}")
    return examples


def plot_confusion_matrix(y_true, y_pred, title, filename):
    """Plots and saves confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5), dpi=300)
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=["Neg (-1)", "Pos (+1)"],
        yticklabels=["Neg (-1)", "Pos (+1)"],
    )
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_SAVE_DIR, filename))
    plt.close()


def plot_loss_curve(losses, title, filename):
    """Plots and saves training loss curve."""
    plt.figure(figsize=(8, 5), dpi=300)
    plt.plot(losses, label="Training Loss", color="red", linewidth=2)
    plt.xlabel("Iterations")
    plt.ylabel("Hinge Loss")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_SAVE_DIR, filename))
    plt.close()


# ==============================================================================
# 5. Dynamic Feature Extraction Strategies
# ==============================================================================
class FeatureExtractorStrategy(ABC):
    """Abstract base class for feature extraction strategies."""

    @abstractmethod
    def fit_transform(self, texts, params=None):
        """Fit extractor and transform training texts."""
        pass

    @abstractmethod
    def transform(self, texts):
        """Transform new texts using fitted extractor."""
        pass


class SklearnDynamicExtractor(FeatureExtractorStrategy):
    """Handles CountVectorizer and TF-IDF extraction with dynamic hyperparameters."""

    def __init__(self, method="count"):
        self.method = method
        self.vectorizer = None

    def fit_transform(self, texts, params=None):
        if params is None:
            params = {}

        max_features = params.get(f"{self.method}_max_features", 5000)
        ngram_range = params.get(f"{self.method}_ngram_range", (1, 2))
        min_df = params.get(f"{self.method}_min_df", 2)

        if self.method == "count":
            self.vectorizer = CountVectorizer(
                stop_words="english",
                max_features=max_features,
                ngram_range=ngram_range,
                min_df=min_df,
            )
        else:
            self.vectorizer = TfidfVectorizer(
                stop_words="english",
                max_features=max_features,
                ngram_range=ngram_range,
                min_df=min_df,
            )

        return self.vectorizer.fit_transform(texts)

    def transform(self, texts):
        if self.vectorizer is None:
            raise ValueError("Extractor not fitted yet.")
        return self.vectorizer.transform(texts)


class GensimDynamicExtractor(FeatureExtractorStrategy):
    """
    Handles Word2Vec (with Global Cache support) and Doc2Vec.
    Supports both pre-trained GloVe vectors and training from scratch.
    """

    def __init__(self, method="word2vec"):
        self.method = method
        self.model = None
        self.vector_size = 300
        self.use_pretrained = method == "word2vec"

    def _tokenize(self, texts):
        """Tokenizes texts into lowercase word lists."""
        return [text.lower().split() for text in texts]

    def _load_pretrained_vectors(self):
        """Attempts to load GloVe vectors into global memory (once)."""
        global GLOBAL_GLOVE_VECTORS

        # Already loaded during startup check
        if GLOBAL_GLOVE_VECTORS is not None:
            return

        print("  [Info] Loading 'glove-wiki-gigaword-300' into memory (ONCE)...")
        try:
            GLOBAL_GLOVE_VECTORS = api.load("glove-wiki-gigaword-300")
            print("  [Success] Pre-trained vectors loaded into memory.")
        except Exception as e:
            print(f"  [Warning] Download/Load failed: {e}")
            print("  [Info] Switching to training from scratch.")
            self.use_pretrained = False

    def fit_transform(self, texts, params=None):
        if params is None:
            params = {}
        tokenized = self._tokenize(texts)

        # Logic for Word2Vec
        if self.method == "word2vec":
            if self.use_pretrained:
                self._load_pretrained_vectors()

            # Case A: Use Pre-trained GloVe (via global cache)
            if self.use_pretrained and GLOBAL_GLOVE_VECTORS is not None:
                vecs = []
                for tokens in tokenized:
                    valid = [
                        GLOBAL_GLOVE_VECTORS[w]
                        for w in tokens
                        if w in GLOBAL_GLOVE_VECTORS
                    ]
                    if valid:
                        vecs.append(np.mean(valid, axis=0))
                    else:
                        vecs.append(np.zeros(self.vector_size))
                return np.array(vecs)

            # Case B: Train Word2Vec from scratch
            else:
                self.vector_size = params.get(f"{self.method}_vector_size", 300)
                window = params.get(f"{self.method}_window", 5)
                epochs = params.get(f"{self.method}_epochs", 10)

                self.model = Word2Vec(
                    sentences=tokenized,
                    vector_size=self.vector_size,
                    window=window,
                    min_count=2,
                    workers=8,
                    seed=SEED,
                    epochs=epochs,
                )
                vecs = []
                for tokens in tokenized:
                    valid = [self.model.wv[w] for w in tokens if w in self.model.wv]
                    if valid:
                        vecs.append(np.mean(valid, axis=0))
                    else:
                        vecs.append(np.zeros(self.vector_size))
                return np.array(vecs)

        # Logic for Doc2Vec
        else:
            self.vector_size = params.get(f"{self.method}_vector_size", 300)
            # Increase epochs for training on small data
            epochs = params.get(f"{self.method}_epochs", 50)

            # Use Integer tags for correct lookups in model.dv
            tagged = [
                TaggedDocument(words=t, tags=[i]) for i, t in enumerate(tokenized)
            ]
            self.model = Doc2Vec(
                vector_size=self.vector_size,
                window=5,
                min_count=2,
                workers=8,
                dm=0,  # Use PV-DBOW, often better for classification
                seed=SEED,
                epochs=epochs,
            )
            self.model.build_vocab(tagged)
            self.model.train(
                tagged, total_examples=self.model.corpus_count, epochs=epochs
            )

            # Correct Extraction: Use learned vectors directly from model.dv
            # This is much more accurate than running inference on training data
            return np.array([self.model.dv[i] for i in range(len(tokenized))])

    def transform(self, texts):
        tokenized = self._tokenize(texts)

        if self.method == "word2vec":
            # Use pre-trained GloVe if available
            if self.use_pretrained and GLOBAL_GLOVE_VECTORS is not None:
                vecs = []
                for tokens in tokenized:
                    valid = [
                        GLOBAL_GLOVE_VECTORS[w]
                        for w in tokens
                        if w in GLOBAL_GLOVE_VECTORS
                    ]
                    if valid:
                        vecs.append(np.mean(valid, axis=0))
                    else:
                        vecs.append(np.zeros(self.vector_size))
                return np.array(vecs)
            # Use trained Word2Vec model
            else:
                vecs = []
                for tokens in tokenized:
                    valid = [self.model.wv[w] for w in tokens if w in self.model.wv]
                    if valid:
                        vecs.append(np.mean(valid, axis=0))
                    else:
                        vecs.append(np.zeros(self.vector_size))
                return np.array(vecs)
        # Use Doc2Vec inference
        else:
            # Increase inference epochs to ensure convergence
            return np.array([self.model.infer_vector(t, epochs=50) for t in tokenized])


class BertStaticExtractor(FeatureExtractorStrategy):
    """Uses pre-trained BERT-Base-Uncased with global caching for efficiency."""

    def __init__(self, batch_size=32):
        self.batch_size = batch_size
        self.available = False

        # Check if BERT was successfully loaded during startup
        global GLOBAL_BERT_MODEL, GLOBAL_BERT_TOKENIZER

        if GLOBAL_BERT_MODEL is not None and GLOBAL_BERT_TOKENIZER is not None:
            self.tokenizer = GLOBAL_BERT_TOKENIZER
            self.model = GLOBAL_BERT_MODEL
            self.available = True
        else:
            print("  [BERT] Not available - skipping BERT extraction")

    def _embed(self, texts):
        """Generates BERT embeddings for input texts."""
        if not self.available:
            raise RuntimeError("BERT model not available")

        all_embeddings = []
        torch.cuda.empty_cache()

        for i in tqdm(range(0, len(texts), self.batch_size), desc="BERT Embedding"):
            batch_texts = texts[i : i + self.batch_size]
            inputs = self.tokenizer(
                batch_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=128,
            )
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = self.model(**inputs)
                # Use [CLS] token embedding
                all_embeddings.append(outputs.last_hidden_state[:, 0, :].cpu().numpy())

            del inputs, outputs

        torch.cuda.empty_cache()
        return np.vstack(all_embeddings)

    def fit_transform(self, texts, params=None):
        """Extract training embeddings (cached globally)."""
        if not self.available:
            raise RuntimeError("BERT model not available")

        global GLOBAL_BERT_TRAIN_EMBEDDINGS

        if GLOBAL_BERT_TRAIN_EMBEDDINGS is not None:
            return GLOBAL_BERT_TRAIN_EMBEDDINGS

        print("  [BERT] Extracting Training Embeddings (computing once)...")
        GLOBAL_BERT_TRAIN_EMBEDDINGS = self._embed(texts)
        return GLOBAL_BERT_TRAIN_EMBEDDINGS

    def transform(self, texts):
        """Extract test embeddings (cached globally)."""
        if not self.available:
            raise RuntimeError("BERT model not available")

        global GLOBAL_BERT_TEST_EMBEDDINGS

        if GLOBAL_BERT_TEST_EMBEDDINGS is not None:
            return GLOBAL_BERT_TEST_EMBEDDINGS

        print("  [BERT] Extracting Test Embeddings (computing once)...")
        GLOBAL_BERT_TEST_EMBEDDINGS = self._embed(texts)
        return GLOBAL_BERT_TEST_EMBEDDINGS

    def precompute_test(self, texts):
        """Pre-compute test embeddings before optimization loop."""
        if not self.available:
            return

        global GLOBAL_BERT_TEST_EMBEDDINGS
        if GLOBAL_BERT_TEST_EMBEDDINGS is None:
            print("  [BERT] Pre-computing Test Embeddings...")
            GLOBAL_BERT_TEST_EMBEDDINGS = self._embed(texts)


# ==============================================================================
# 6. Classifiers
# ==============================================================================
class ManualHingeSGD:
    """
    Manual implementation of Hinge Loss SGD Linear Classifier.
    Supports learning rate decay and L2 regularization.
    """

    def __init__(self, num_iters=100, eta=0.01, lam=1e-4, decay=True):
        self.num_iters = num_iters
        self.eta = eta
        self.lam = lam
        self.decay = decay
        self.weights = None
        self.loss_history = []

    def fit(self, X, y):
        """Train the classifier using Hinge Loss SGD."""
        # Convert sparse matrix to dense if needed
        if hasattr(X, "toarray"):
            X = X.toarray()

        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)

        # Convert labels to {-1, +1}
        y_mod = np.where(y <= 0, -1, 1)

        current_eta = self.eta

        for epoch in range(self.num_iters):
            # Apply learning rate decay
            if self.decay:
                current_eta = self.eta / (1 + 0.01 * epoch)

            epoch_loss = 0

            # Shuffle data for SGD
            indices = np.random.permutation(n_samples)
            X_shuffled = X[indices]
            y_shuffled = y_mod[indices]

            # Process each sample
            for i in range(n_samples):
                xi = X_shuffled[i]
                yi = y_shuffled[i]
                score = np.dot(self.weights, xi)

                # Hinge loss gradient
                if yi * score < 1:
                    grad = self.lam * self.weights - yi * xi
                    epoch_loss += 1 - yi * score
                else:
                    grad = self.lam * self.weights

                # Update weights
                self.weights -= current_eta * grad

            # Record epoch loss (with regularization term)
            epoch_loss += (self.lam / 2) * np.sum(self.weights**2)
            self.loss_history.append(epoch_loss / n_samples)

    def predict(self, X):
        """Predict class labels for samples in X."""
        if hasattr(X, "toarray"):
            X = X.toarray()
        return np.where(np.dot(X, self.weights) >= 0, 1, -1)

    def save(self, path):
        """Save model to disk."""
        joblib.dump(self, path)


# ==============================================================================
# 7. Joint Optimization Pipeline
# ==============================================================================
class JointOptimizationPipeline:
    """
    Unified pipeline for joint feature extraction and classifier optimization.
    Supports Optuna-based hyperparameter tuning with AUC optimization.
    """

    def __init__(self, feature_name, classifier_name, n_trials=60):
        self.feature_name = feature_name
        self.classifier_name = classifier_name
        self.n_trials = n_trials
        self.scaler = None  # Add scaler for Doc2Vec/Word2Vec + HingeSGD

        # Initialize feature extractor
        if feature_name in ["CountVec", "TF-IDF"]:
            method = "count" if feature_name == "CountVec" else "tfidf"
            self.extractor = SklearnDynamicExtractor(method)
        elif feature_name in ["Word2Vec", "Doc2Vec"]:
            method = feature_name.lower()
            self.extractor = GensimDynamicExtractor(method)
        elif feature_name == "BERT":
            self.extractor = BertStaticExtractor()
            if not self.extractor.available:
                raise RuntimeError("BERT not available - cannot create pipeline")

        self.best_model = None
        self.best_params = None
        self.loss_history = []

        # Cache for expensive feature computations
        self.precomputed_X_sub = None
        self.precomputed_X_val = None
        self.precomputed_X_full = None

    def _get_optuna_params(self, trial):
        """Define hyperparameter search space for Optuna."""
        params = {}

        # Feature extraction hyperparameters (only for sklearn vectorizers)
        if self.feature_name in ["CountVec", "TF-IDF"]:
            key = "count" if self.feature_name == "CountVec" else "tfidf"
            params[f"{key}_max_features"] = trial.suggest_int(
                f"{key}_max_features", 1000, 20000
            )
            params[f"{key}_min_df"] = trial.suggest_int(f"{key}_min_df", 2, 5)
            ngram = trial.suggest_categorical(f"{key}_ngram", ["uni", "bi"])
            params[f"{key}_ngram_range"] = (1, 1) if ngram == "uni" else (1, 2)

        # Classifier hyperparameters
        if self.classifier_name == "SVM":
            params["svm_C"] = trial.suggest_float("svm_C", 1e-3, 100, log=True)
            params["svm_kernel"] = trial.suggest_categorical(
                "svm_kernel", ["linear", "rbf"]
            )

        elif self.classifier_name == "LogisticRegression":
            params["lr_C"] = trial.suggest_float("lr_C", 1e-4, 100, log=True)
            params["lr_penalty"] = trial.suggest_categorical("lr_penalty", ["l1", "l2"])

        return params

    def run(self, X_raw_train, y_train, X_raw_test):
        """Execute the complete training pipeline."""

        # Pre-compute BERT test embeddings if applicable
        if isinstance(self.extractor, BertStaticExtractor):
            self.extractor.precompute_test(X_raw_test)

        # Case 1: HingeSGD (no hyperparameter tuning)
        if self.classifier_name == "HingeSGD":
            print("  [HingeSGD] Training directly without tuning...")
            default_params = {}

            # Set reasonable defaults for Word2Vec/Doc2Vec
            if self.feature_name in ["Word2Vec", "Doc2Vec"]:
                default_params = {
                    f"{self.feature_name.lower()}_vector_size": 300,
                    f"{self.feature_name.lower()}_epochs": 50
                    if self.feature_name == "Doc2Vec"
                    else 10,
                }

            # Use smaller learning rate for high-dimensional CountVec to prevent explosion
            eta = 0.001 if self.feature_name == "CountVec" else 0.01

            model = ManualHingeSGD(num_iters=100, eta=eta, decay=True)
            X_vec = self.extractor.fit_transform(X_raw_train, default_params)

            # --- Scaling for Doc2Vec/Word2Vec with HingeSGD ---
            if self.feature_name in ["Word2Vec", "Doc2Vec"]:
                self.scaler = StandardScaler()
                X_vec = self.scaler.fit_transform(X_vec)
            # --------------------------------------------------

            model.fit(X_vec, y_train)

            self.best_model = model
            self.loss_history = model.loss_history
            return

        # Case 2: Optuna-based hyperparameter tuning (for SVM and LogisticRegression)
        print(
            f"  [Optuna] Tuning {self.feature_name} + {self.classifier_name} "
            f"(optimizing AUC with {self.n_trials} trials)..."
        )

        # Split training data for validation
        X_sub, X_val, y_sub, y_val = train_test_split(
            X_raw_train, y_train, test_size=0.2, random_state=SEED, stratify=y_train
        )

        # Determine if feature extraction is expensive (to enable caching)
        is_expensive_feature = self.feature_name in ["Word2Vec", "Doc2Vec", "BERT"]

        # Pre-compute features for expensive extractors
        if is_expensive_feature:
            print("  [Info] Pre-computing features to accelerate tuning...")
            static_params = {}

            # Use fixed parameters during tuning for Word2Vec/Doc2Vec
            if self.feature_name in ["Word2Vec", "Doc2Vec"]:
                static_params = {
                    f"{self.feature_name.lower()}_vector_size": 300,
                    f"{self.feature_name.lower()}_window": 5,
                    f"{self.feature_name.lower()}_epochs": 50
                    if self.feature_name == "Doc2Vec"
                    else 10,
                }

            self.precomputed_X_sub = self.extractor.fit_transform(X_sub, static_params)
            self.precomputed_X_val = self.extractor.transform(X_val)
            self.precomputed_X_full = self.extractor.fit_transform(
                X_raw_train, static_params
            )

        def objective(trial):
            """Optuna objective function (maximize AUC)."""
            params = self._get_optuna_params(trial)

            # Step 1: Feature extraction
            if is_expensive_feature:
                X_train_vec = self.precomputed_X_sub
                X_valid_vec = self.precomputed_X_val
            else:
                X_train_vec = self.extractor.fit_transform(X_sub, params)
                X_valid_vec = self.extractor.transform(X_val)

            # Step 2: Train classifier and compute AUC
            auc_score = 0.5  # Default fallback value

            try:
                if self.classifier_name == "SVM":
                    if params["svm_kernel"] == "linear":
                        # Use LinearSVC for efficiency with linear kernel
                        clf = LinearSVC(
                            C=params["svm_C"], random_state=SEED, dual=False
                        )
                        clf.fit(X_train_vec, y_sub)
                        y_scores = clf.decision_function(X_valid_vec)
                    else:
                        # Use standard SVC for RBF kernel
                        clf = SVC(
                            C=params["svm_C"],
                            kernel="rbf",
                            gamma="scale",
                            random_state=SEED,
                        )
                        clf.fit(X_train_vec, y_sub)
                        y_scores = clf.decision_function(X_valid_vec)

                    auc_score = roc_auc_score(y_val, y_scores)

                elif self.classifier_name == "LogisticRegression":
                    clf = LogisticRegression(
                        C=params["lr_C"],
                        penalty=params["lr_penalty"],
                        solver="liblinear",
                        random_state=SEED,
                        max_iter=1000,
                    )
                    clf.fit(X_train_vec, y_sub)
                    y_scores = clf.predict_proba(X_valid_vec)[:, 1]
                    auc_score = roc_auc_score(y_val, y_scores)

            except Exception:
                # Return baseline AUC on failure
                auc_score = 0.5

            return auc_score

        # Run Optuna optimization
        study = optuna.create_study(direction="maximize")
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        study.optimize(objective, n_trials=self.n_trials)

        self.best_params = study.best_params
        print(f"  >> Best Hyperparameters: {self.best_params}")

        # Step 3: Refit on full training data with best parameters
        print("  >> Refitting on full training data...")
        if is_expensive_feature:
            X_full_vec = self.precomputed_X_full
        else:
            X_full_vec = self.extractor.fit_transform(X_raw_train, self.best_params)

        if self.classifier_name == "SVM":
            if self.best_params["svm_kernel"] == "linear":
                self.best_model = LinearSVC(
                    C=self.best_params["svm_C"], random_state=SEED, dual=False
                )
            else:
                self.best_model = SVC(
                    C=self.best_params["svm_C"],
                    kernel="rbf",
                    gamma="scale",
                    probability=True,
                    random_state=SEED,
                )
            self.best_model.fit(X_full_vec, y_train)

        elif self.classifier_name == "LogisticRegression":
            self.best_model = LogisticRegression(
                C=self.best_params["lr_C"],
                penalty=self.best_params["lr_penalty"],
                solver="liblinear",
                random_state=SEED,
                max_iter=3000,
            )
            self.best_model.fit(X_full_vec, y_train)

    def evaluate(self, X_raw_test, y_test):
        """Evaluate the trained model on test data."""
        X_test_vec = self.extractor.transform(X_raw_test)

        # --- Apply scaling if used (Doc2Vec/Word2Vec + HingeSGD) ---
        if self.scaler is not None:
            X_test_vec = self.scaler.transform(X_test_vec)
        # -----------------------------------------------------------

        y_pred = self.best_model.predict(X_test_vec)

        # Compute standard metrics
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
        rec = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
        f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)

        # Compute AUC with proper score extraction
        try:
            if hasattr(self.best_model, "predict_proba"):
                # For Logistic Regression / SVC with probability=True
                y_scores = self.best_model.predict_proba(X_test_vec)[:, 1]
            elif hasattr(self.best_model, "decision_function"):
                # For LinearSVC / SVC without probability
                y_scores = self.best_model.decision_function(X_test_vec)
            elif hasattr(self.best_model, "weights"):
                # For Manual HingeSGD
                if hasattr(X_test_vec, "toarray"):
                    X_test_vec = X_test_vec.toarray()
                y_scores = np.dot(X_test_vec, self.best_model.weights)
            else:
                y_scores = y_pred  # Fallback to predictions

            auc = roc_auc_score(y_test, y_scores)
        except Exception as e:
            print(f"  [Warning] AUC calculation failed: {e}")
            auc = 0.5

        return y_pred, acc, prec, rec, f1, auc

    def save_artifacts(self, filename):
        """Save the trained model to disk."""
        joblib.dump(self.best_model, filename)


# ==============================================================================
# 8. Main Experiment Execution
# ==============================================================================
def run_experiment():
    """Main experiment orchestration function."""

    # Pre-download all models at startup
    download_all_pretrained_models()

    # Define dataset paths
    train_path = "data/data_rt.train"
    test_path = "data/data_rt.test"

    print("\n>>> 1. Loading Datasets...")
    full_train_data = read_examples(train_path)
    test_data = read_examples(test_path)

    if not full_train_data or not test_data:
        print("[Fatal Error] No data loaded. Please check 'data/' directory.")
        return

    X_train_raw = [x[0] for x in full_train_data]
    y_train_raw = np.array([x[1] for x in full_train_data])
    X_test_raw = [x[0] for x in test_data]
    y_test_raw = np.array([x[1] for x in test_data])

    # Define experiment configurations
    feature_types = ["CountVec", "TF-IDF", "Word2Vec", "Doc2Vec"]

    # Add BERT only if available
    if GLOBAL_BERT_MODEL is not None:
        feature_types.append("BERT")

    classifier_types = ["HingeSGD", "LogisticRegression", "SVM"]

    results = []

    print("\n>>> 2. Starting Joint Optimization Loop...")

    for feat_name in feature_types:
        for clf_name in classifier_types:
            print(f"\n{'=' * 60}")
            print(f"Pipeline: {feat_name} + {clf_name}")
            print(f"{'=' * 60}")

            try:
                pipeline = JointOptimizationPipeline(feat_name, clf_name, n_trials=60)

                t0 = time.time()
                pipeline.run(X_train_raw, y_train_raw, X_test_raw)
                train_time = time.time() - t0

                # Evaluation
                y_pred, acc, prec, rec, f1, auc = pipeline.evaluate(
                    X_test_raw, y_test_raw
                )
                print(
                    f"  >> [TEST] Acc: {acc:.4f} | F1: {f1:.4f} | "
                    f"AUC: {auc:.4f} | Time: {train_time:.1f}s"
                )

                # Record results
                best_params_str = (
                    str(pipeline.best_params) if pipeline.best_params else "Default"
                )
                results.append(
                    {
                        "Feature": feat_name,
                        "Classifier": clf_name,
                        "Accuracy": acc,
                        "Precision": prec,
                        "Recall": rec,
                        "F1-Score": f1,
                        "ROC-AUC": auc,
                        "Best_Params": best_params_str,
                    }
                )

                # Save artifacts
                base_name = f"{feat_name}_{clf_name}"
                pipeline.save_artifacts(
                    os.path.join(MODEL_SAVE_DIR, f"model_{base_name}.pkl")
                )

                plot_confusion_matrix(
                    y_test_raw,
                    y_pred,
                    f"CM: {feat_name}+{clf_name}",
                    f"cm_{base_name}.png",
                )

                if clf_name == "HingeSGD" and pipeline.loss_history:
                    plot_loss_curve(
                        pipeline.loss_history,
                        f"Loss: {feat_name}",
                        f"loss_{base_name}.png",
                    )

            except Exception as e:
                print(f"  [Error] Pipeline failed: {e}")
                continue

            # Memory cleanup
            gc.collect()

    # Generate final report
    df_res = pd.DataFrame(results)

    print("\n\n" + "#" * 60)
    print("PERFORMANCE RANKING")
    print("#" * 60)

    print("\n>>> Sorted by Accuracy:")
    print(
        df_res.sort_values(by="Accuracy", ascending=False)[
            ["Feature", "Classifier", "Accuracy", "F1-Score", "ROC-AUC"]
        ].to_markdown(index=False)
    )

    df_res.to_csv("experiment_results.csv", index=False)
    print(
        "\n[Done] Results saved to 'experiment_results.csv'. "
        "Check 'plots/' for visualizations."
    )


if __name__ == "__main__":
    run_experiment()